In [18]:
import pandas as pd

df_ranked = pd.read_csv('./ranked.csv', usecols=["displayName", "summaryType", "primaryTypeDisplayName", "rating", "userRatingCount",
    "wilson_score", "wilson_quantile", "takeaway", "shortFormattedAddress"])

In [19]:
df_ranked[df_ranked["summaryType"]=="Restaurant"]

,displayName,primaryTypeDisplayName,rating,userRatingCount,shortFormattedAddress,summaryType,takeaway,wilson_score,wilson_quantile
2,Mate’s Clapham Restaurant,Restaurant,4.9,341.0,"3 Clapham Common South Side, London",Restaurant,Dine-In,0.952401,0.988571
48,Upstairs,Restaurant,4.7,102.0,"4 The Polygon, London",Restaurant,Dine-In,0.857077,0.725714
79,Soul Lounge,Restaurant,4.3,866.0,"76 Clapham High St, London",Restaurant,Dine-In,0.798273,0.548571
85,POTAGE,Restaurant,4.5,68.0,"Unit 7, Lindford Street Business Estate, 2 Lin...",Restaurant,Dine-In,0.775886,0.514286
91,Big Taste Battersea,Restaurant,4.8,20.0,"V2, Railway Arches, Arch, Patcham Terrace, London",Restaurant,Dine-In,0.763864,0.480000
106,Martine's Bar & Grill,Restaurant,5.0,10.0,"56 Clapham Park Rd, London",Restaurant,Dine-In,0.722460,0.391429
109,Dallas Peri Peri,Restaurant,4.2,119.0,"355 Wandsworth Rd, London",Restaurant,Dine-In,0.719262,0.377143
121,Mike's Kitchen,Restaurant,4.0,81.0,"165 Battersea Park Rd, Greater, London",Restaurant,Dine-In,0.645846,0.308571
156,Zen-G Kitchen,Restaurant,5.0,2.0,"Lindford Street Business Estate, 76 Stewart's ...",Restaurant,Dine-In,0.342372,0.108571
161,Fatchoy canteen（Battersea）,Restaurant,5.0,1.0,"279 Battersea Park Rd, London",Restaurant,Dine-In,0.206543,0.074286


#### Percentile Sort to Type
A global Wilson Score ranking buries niche cuisines (Ethiopian, Korean, etc.) under sheer volume of more common categories. Ranking each `primaryType` independently and then expressing the result as a **within-type percentile** fixes this — a hidden gem at the top of its niche surfaces clearly.

Two new columns are added:
- **`type_rank`** — position within the `primaryType` group (1 = best)
- **`type_percentile`** — how far the place sits in its category on a 0–100 scale (100 = top of its type)

In [20]:
from scipy.stats import percentileofscore

# Within each primaryType, rank by wilson_score and compute a percentile.
# percentileofscore uses "rank" kind → ties get the average of their positions.
TYPE = "summaryType"
df_ranked["type_rank"] = (
    df_ranked.groupby(TYPE)["wilson_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)
# display(df_ranked)
df_ranked["type_percentile"] = df_ranked.groupby(TYPE)["wilson_score"].transform(
    lambda s: s.apply(lambda v: round(percentileofscore(s, v, kind="rank"), 1))
)

# ── Summary: how many places per type, and the top entry in each ─────────────
type_summary = (
    df_ranked.groupby(TYPE)
    .apply(lambda g: g.nlargest(1, "wilson_score")[
        ["displayName", "rating", "userRatingCount", "wilson_score", "type_percentile"]
    ], include_groups=False)
    .reset_index(level=1, drop=True)
    .join(df_ranked.groupby(TYPE).size().rename("count"))
    .sort_values("count", ascending=False)
).reset_index()

print(f"Unique types: {df_ranked[TYPE].nunique()}")
type_summary[["summaryType", "count"]]


Unique types: 28


,summaryType,count
0,European,28
1,Fast Food,26
2,Pizza,18
3,Restaurant,14
4,Brunch & Breakfast,9
5,Japanese,7
6,Burgers,7
7,Chinese,6
8,Latin American,6
9,South Asian,6


In [7]:
# ── Inspect top-ranked places within a specific type ─────────────────────────
SHOW_TYPE = "restaurant"   # ← change to any primaryType in the dataset

(
    df_ranked[df_ranked[TYPE] == SHOW_TYPE]
    .sort_values("type_rank")
    [["type_rank", "displayName", "rating", "userRatingCount",
      "wilson_score", "type_percentile", "shortFormattedAddress"]]
    .head(20)
)


,type_rank,displayName,rating,userRatingCount,wilson_score,type_percentile,shortFormattedAddress
